# Diffraction reflections — new layout (dev)

Development version of `apps/diffraction_reflections.ipynb` with an arrangeable
interface: a layout switch (large 3D / side by side / stacked), collapsible
panels, tabbed inputs, the tools inside the app and the instructions behind
the **help** button. The calculations are unchanged.


In [ ]:
# Help text shown by the app's "help" button (kept here so the page opens on the
# app itself rather than below a long introduction).
HELP_HTML = """
<div style="font-size:12.5px;line-height:1.45;max-width:1100px">
<p><b>What it does.</b> Diffractometer angles for an (h,k,l) reflection with
diffcalc's <code>you</code> geometry. The beamline calls the out-of-plane
detector circle <b>gamma</b>; diffcalc calls it <code>nu</code> &mdash; the
rename happens automatically.</p>

<p><b>Axes.</b> Lab directions (the xyz half of an orientation line) are in
<b>SwissFEL axes</b>: x horizontal transverse, y up, z along the beam. The
surface normal is an (h,k,l), so it has no lab frame.</p>

<p><b>Solving.</b> Tick <i>fix</i> for the angles you hold; diffcalc solves the
rest. Several solution branches can exist: they appear as columns
<code>sol. 0</code>, <code>sol. 1</code>, &hellip; The <i>min/max</i> boxes beside
each angle drop branches outside that range (empty = no limit). The six
physical circles are shown; diffcalc's virtual constraints (qaz, naz, alpha,
beta, psi, &hellip;) are under <i>more constraints</i>.</p>

<p><b>3D view.</b> Click a solution column to drive the mechanism.
<b>config</b> (the default) shows the angle values entered, even when no
solution exists; <b>zero</b> returns every circle to 0. The view's own
options sit in its panel: reciprocal vectors k, k&prime; and Q = k&prime; &minus; k;
the reciprocal cell a*, b*, c*; the <i>measured</i> hkl of the current pose
(computed live from UB and the circles &mdash; integer only on a Bragg condition);
the Ewald sphere (radius |k|) and the patch of it the detector sees.
Reciprocal lengths use diffcalc's 2&pi; convention (|k| = 2&pi;/&lambda;,
|Q| = 2&pi;/d).</p>

<p><b>Layout.</b> <i>large 3D</i> (the default), <i>side by side</i> or <i>stacked</i>
rearrange the panels; &#9662;/&#9656; collapses one; <i>3D height</i> sizes the
view without resetting it.</p>

<p><b>Config.</b> <i>load preset</i> applies a stored beamline configuration
(angles, fix conditions, filters only &mdash; sample and hkl are kept).
<i>store config&hellip;</i> writes the inputs to a JSON file; untick groups to
leave them out, so a file can be a partial template. It can include the 3D
view's display settings and this panel layout.</p>

<p><b>Tools.</b> <i>table of reflections</i> solves a list of h,k,l with the
current sample and constraints; <i>detector footprint</i> gives the Q / hkl
range a flat detector covers at a given distance and size.</p>
</div>
"""


import base64
import html
import json
import sys
from pathlib import Path

# the repo root is the nearest folder holding bernina_prep, wherever this runs
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bernina_prep").is_dir())
try:
    import bernina_prep  # noqa: F401
except ImportError:
    sys.path.insert(0, str(REPO))

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from bernina_prep.diffraction import (
    build_ubcalc,
    k_magnitude,
    make_constraints,
    reciprocal_basis,
    solution_angles,
    solve_hkl,
    surface_normal_phi,
)

# Narrower value fields than before, to leave room for the per-angle solution
# filters next to them; filters are half that width again.
UC_W = "150px"
FIELD_W = "98px"
FILTER_W = "49px"
HKL_W = "98px"
DESC_W = "58px"
LABEL_W = "82px"
HKL_DECIMALS = 4

LAT_NAME = "custom lattice"

# This applet speaks SwissFEL lab axes throughout: x horizontal transverse,
# y vertical, z along the beam. diffcalc uses the same right-handed frame with
# a cyclic relabelling (diffcalc +x,+y,+z = SwissFEL +y,+z,+x), and
# bernina_prep.diffraction converts at its boundary.
LAB_FRAME = "swissfel"

# One standard ordering, used for the inputs AND the solution rows: the six
# physical circles first, then diffcalc's virtual/derived angles.
PHYSICAL_ANGLES = ["gam", "delta", "mu", "eta", "chi", "phi"]
VIRTUAL_ANGLES = ["qaz", "naz", "alpha", "beta", "psi", "a_eq_b", "mu_is_gam"]
angle_names = PHYSICAL_ANGLES + VIRTUAL_ANGLES
BOOL_ANGLES = {"a_eq_b", "mu_is_gam"}

# solve_hkl reports 'gam' as 'gamma'; same order, plus the derived quantities.
ROW_ORDER = ["gamma", "delta", "mu", "eta", "chi", "phi",
             "theta", "ttheta", "qaz", "naz", "alpha", "beta", "psi",
             "tau", "betain", "betaout"]

default_fixed = {"eta", "chi", "phi"}

ctrl = {
    "uc_a": widgets.FloatText(description="a (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_b": widgets.FloatText(description="b (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_c": widgets.FloatText(description="c (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_alpha": widgets.FloatText(description="alpha", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_beta": widgets.FloatText(description="beta", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_gamma": widgets.FloatText(description="gamma", value=90, layout=widgets.Layout(width=UC_W)),
    # float hkl (a reflection need not sit on an integer node), stepping by 0.1
    # on the spinner and rounded to HKL_DECIMALS so the field never fills up
    # with floating-point noise
    "h": widgets.FloatText(value=1, step=0.1, description="h",
                           layout=widgets.Layout(width=HKL_W),
                           style={"description_width": "20px"}),
    "k": widgets.FloatText(value=1, step=0.1, description="k",
                           layout=widgets.Layout(width=HKL_W),
                           style={"description_width": "20px"}),
    "l": widgets.FloatText(value=1, step=0.1, description="l",
                           layout=widgets.Layout(width=HKL_W),
                           style={"description_width": "20px"}),
    # Lab-frame directions are entered in SwissFEL axes (x transverse, y up,
    # z beam) and converted to diffcalc's on the way in - see LAB_FRAME below.
    # The surface normal is an (h,k,l), so it carries no lab frame at all.
    "surface_normal": widgets.Text(value="(1,1,1)",
                                   description="surf. normal (h,k,l)",
                                   layout=widgets.Layout(width="372px"),
                                   style={"description_width": "168px"}),
    "orientations": widgets.Textarea(
        value="(1,1,1) > (1,0,0)\n(1,1,-1) > (0,1,0)",
        description="orientations (h,k,l) > (x,y,z)",
        placeholder='one "(h,k,l) > (x,y,z)" pair per line, xyz in SwissFEL axes',
        layout=widgets.Layout(width="372px", height="66px"),
        style={"description_width": "168px"},
    ),
}

# per-angle solution filters; an empty box means "no constraint"
filters = {}
angle_rows = []
for name in angle_names:
    if name in BOOL_ANGLES:
        ctrl[name] = widgets.Checkbox(value=False, indent=False,
                                      layout=widgets.Layout(width=FIELD_W))
    else:
        ctrl[name] = widgets.FloatText(value=0, layout=widgets.Layout(width=FIELD_W))
    ctrl[f"{name}_fixed"] = widgets.Checkbox(value=name in default_fixed,
                                             description="fix", indent=False,
                                             layout=widgets.Layout(width="62px"))
    row = [widgets.Label(name, layout=widgets.Layout(width=LABEL_W)), ctrl[name]]
    if name in BOOL_ANGLES:
        row.append(widgets.Label("", layout=widgets.Layout(width="106px")))
    else:
        lo = widgets.Text(placeholder="min", layout=widgets.Layout(width=FILTER_W))
        hi = widgets.Text(placeholder="max", layout=widgets.Layout(width=FILTER_W))
        filters[name] = (lo, hi)
        row += [lo, hi]
    row.append(ctrl[f"{name}_fixed"])
    angle_rows.append(widgets.HBox(row))

ctrl["energy"] = widgets.FloatSlider(value=8000, min=1000, max=20000, step=10,
                                     readout=False, continuous_update=False,
                                     layout=widgets.Layout(width="170px"))
energy_value = widgets.BoundedFloatText(description="energy", value=8000, min=1000,
                                        max=20000, step=10,
                                        layout=widgets.Layout(width="150px"),
                                        style={"description_width": DESC_W})
widgets.link((energy_value, "value"), (ctrl["energy"], "value"))
energy_row = widgets.HBox([energy_value, widgets.Label("eV"), ctrl["energy"]])

ctr_uc = widgets.HBox([ctrl[n] for n in ("uc_a", "uc_b", "uc_c", "uc_alpha", "uc_beta", "uc_gamma")],
                      layout=widgets.Layout(flex_flow="row wrap"))
ctr_hkl = widgets.HBox([ctrl["h"], ctrl["k"], ctrl["l"]],
                       layout=widgets.Layout(flex_flow="row wrap"))
ctr_or = widgets.VBox([ctrl["surface_normal"], ctrl["orientations"]])
filter_header = widgets.HTML(
    f'<div style="font-size:11px;color:#666;margin-top:6px">'
    f'<span style="display:inline-block;width:{LABEL_W}"></span>'
    f'<span style="display:inline-block;width:{FIELD_W}">value</span>'
    f'<span style="display:inline-block;width:106px">solution filter</span>'
    f'</div>')
ctr_ang = widgets.VBox([filter_header] + angle_rows)

# --- stored beamline presets ------------------------------------------------
# JSON files in apps/bernina_parameters/. These deliberately apply ONLY the
# angle values, the fix conditions and the solution filters - whatever sample
# and hkl you have entered is left untouched - and then solve once.

PRESET_DIR = REPO / "apps" / "bernina_parameters"
PRESET_GROUPS = ("angles", "fixed", "filters")

# While a whole config is being applied, every widget write would otherwise
# fire its own solve; suspend and run exactly one at the end.
_suspend = {"on": False}


def preset_label(path):
    """Bernina_THC_diffraction_config.json -> 'Bernina THC'."""
    stem = path.stem
    for suffix in ("_diffraction_config", "_config"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
            break
    return stem.replace("_", " ").strip()


def find_presets():
    if not PRESET_DIR.is_dir():
        return {}
    return {preset_label(p): p for p in sorted(PRESET_DIR.glob("*.json"))}


presets = find_presets()
PRESET_NONE = "— preset —"
preset_dd = widgets.Dropdown(
    options=[PRESET_NONE] + list(presets),
    value=PRESET_NONE,
    description="load preset",
    layout=widgets.Layout(width="330px"),
    style={"description_width": "90px"},
)
preset_msg = widgets.HTML()


def on_preset(change):
    name = change["new"]
    if name == PRESET_NONE or name not in presets:
        return
    try:
        cfg = json.loads(presets[name].read_text())
    except Exception as exc:
        preset_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'
        return
    subset = {k: v for k, v in cfg.items() if k in PRESET_GROUPS}
    ignored = [k for k in cfg if k not in PRESET_GROUPS]
    _suspend["on"] = True
    try:
        apply_config(subset)
    finally:
        _suspend["on"] = False
    recompute()                      # exactly one solve after the batch
    note = f"loaded <b>{html.escape(name)}</b>: {', '.join(sorted(subset))}"
    if ignored:
        note += f" (ignored {', '.join(sorted(ignored))}; sample and hkl kept)"
    preset_msg.value = f'<span style="font-size:11px;color:#282">{note}</span>'


preset_dd.observe(on_preset, names="value")


# --- store / recall configurations ------------------------------------------

INCLUDE_GROUPS = [
    ("sample", "sample parameters (unit cell, orientations, surface normal)"),
    ("hkl", "h, k, l"),
    ("energy", "energy"),
    ("angles", "angle values"),
    ("fixed", "fix conditions"),
    ("filters", "solution filters"),
    ("display", "3D view display (toggles, reciprocal scale, detector)"),
    ("layout", "panel layout (arrangement, open panels, 3D height)"),
]
include_boxes = {
    key: widgets.Checkbox(value=True, description=label, indent=False,
                          layout=widgets.Layout(width="330px"))
    for key, label in INCLUDE_GROUPS
}

store_btn = widgets.Button(description="store config…", layout=widgets.Layout(width="130px"))
make_btn = widgets.Button(description="create file", button_style="primary",
                          layout=widgets.Layout(width="110px"))
file_name = widgets.Text(value="bernina_diffraction_config", description="file name",
                         layout=widgets.Layout(width="330px"),
                         style={"description_width": "70px"})
upload = widgets.FileUpload(accept=".json", multiple=False,
                            description="load config", layout=widgets.Layout(width="150px"))
download_link = widgets.HTML()
config_msg = widgets.HTML()
store_panel = widgets.VBox(
    [widgets.HTML('<div style="font-size:11px;color:#666">include in the file '
                  '(unchecked groups are left out, so the file acts as a partial '
                  'template and anything missing stays free on load):</div>')]
    + [include_boxes[k] for k, _ in INCLUDE_GROUPS]
    + [file_name, widgets.HBox([make_btn]), download_link],
    layout=widgets.Layout(display="none", border="1px solid #ddd",
                          padding="6px", margin="4px 0 0 0"),
)


def toggle_store(_b):
    shown = store_panel.layout.display != "none"
    store_panel.layout.display = "none" if shown else "flex"


store_btn.on_click(toggle_store)


def current_config():
    cfg = {}
    if include_boxes["sample"].value:
        cfg["sample"] = {n: ctrl[n].value for n in
                         ("uc_a", "uc_b", "uc_c", "uc_alpha", "uc_beta", "uc_gamma",
                          "surface_normal", "orientations")}
    if include_boxes["hkl"].value:
        cfg["hkl"] = {n: ctrl[n].value for n in ("h", "k", "l")}
    if include_boxes["energy"].value:
        cfg["energy"] = ctrl["energy"].value
    if include_boxes["angles"].value:
        cfg["angles"] = {n: ctrl[n].value for n in angle_names}
    if include_boxes["fixed"].value:
        cfg["fixed"] = {n: ctrl[f"{n}_fixed"].value for n in angle_names}
    if include_boxes["filters"].value:
        # kept as strings, empty included on purpose: an empty filter is a
        # meaningful "free" state in a template
        cfg["filters"] = {n: [lo.value, hi.value] for n, (lo, hi) in filters.items()}
    if include_boxes["layout"].value:
        cfg["layout"] = layout_state()
    return cfg


def config_filename():
    """The entered name, made safe for a download and ending in .json."""
    stem = file_name.value.strip()
    if stem.lower().endswith(".json"):
        stem = stem[:-5]
    stem = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in stem).strip("._")
    return (stem or "bernina_diffraction_config") + ".json"


def on_make(_b):
    cfg = current_config()
    with_display = include_boxes["display"].value
    if not cfg and not with_display:
        download_link.value = '<span style="color:#b00">nothing selected</span>'
        return
    blob = base64.b64encode(json.dumps(cfg, indent=2).encode()).decode()
    # The display settings live in the 3D page, which python cannot read: with
    # that group ticked, the link carries id + data-cfg and the page keeps its
    # href rewritten to this config plus its current display block.
    hook = (f'id="diff3d-config-download" data-cfg="{blob}" ' if with_display else "")
    name = config_filename()
    download_link.value = (
        f'<a download="{name}" {hook}'
        f'href="data:application/json;base64,{blob}" '
        f'style="font-size:12px">&#11015; download {name}</a>')


make_btn.on_click(on_make)
# a link made before renaming would still carry the old name
file_name.observe(lambda _c: setattr(download_link, "value", ""), names="value")


def apply_config(cfg):
    applied = []
    for name, value in (cfg.get("sample") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    for name, value in (cfg.get("hkl") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    if "energy" in cfg:
        ctrl["energy"].value = cfg["energy"]
        applied.append("energy")
    for name, value in (cfg.get("angles") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    for name, value in (cfg.get("fixed") or {}).items():
        if f"{name}_fixed" in ctrl:
            ctrl[f"{name}_fixed"].value = bool(value)
            applied.append(f"{name}_fixed")
    for name, pair in (cfg.get("filters") or {}).items():
        if name in filters:
            filters[name][0].value = pair[0] or ""
            filters[name][1].value = pair[1] or ""
            applied.append(f"{name}_filter")
    if isinstance(cfg.get("layout"), dict):
        apply_layout(cfg["layout"])
        applied.append("layout")
    if isinstance(cfg.get("display"), dict):
        # rides on the next push only (see push_view)
        state["display"] = cfg["display"]
        applied.append("display")
    return applied


def on_upload(change):
    items = change["new"]
    if not items:
        return
    item = items[0] if isinstance(items, (list, tuple)) else list(items.values())[0]
    try:
        raw = item["content"]
        cfg = json.loads(bytes(raw).decode())
        _suspend["on"] = True
        try:
            applied = apply_config(cfg)
        finally:
            _suspend["on"] = False
        recompute()
        config_msg.value = (f'<span style="font-size:11px;color:#282">loaded '
                            f'{len(applied)} settings; groups absent from the file '
                            f'were left as they are</span>')
    except Exception as exc:
        config_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'


upload.observe(on_upload, names="value")

# --- solutions table, one clickable column per (surviving) branch ------------

ROW_LABEL_W = 70
COL_W = 84
SELECTED_BG = "#cfe8fb"

# "selected" is a solution column index, or one of the two fixed poses below
SEL_CONFIG, SEL_ZERO = "config", "zero"
state = {"df": None, "cols": [], "selected": SEL_CONFIG, "ubcalc": None, "cons": None,
         "display": None}

solution_buttons = widgets.HBox([])
table_html = widgets.HTML()
solve_msg = widgets.HTML()
output = widgets.Output()


def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "NaN"
    return f"{float(v):.3f}"


def _num_or_none(text):
    text = (text or "").strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def passing_columns(df):
    """Columns surviving the per-angle min/max filters (empty box = ignored)."""
    keep = []
    for col in df.columns:
        ok = True
        for name, (lo_w, hi_w) in filters.items():
            row = "gamma" if name == "gam" else name
            if row not in df.index:
                continue
            lo, hi = _num_or_none(lo_w.value), _num_or_none(hi_w.value)
            if lo is None and hi is None:
                continue
            v = df.loc[row, col]
            if pd.isna(v) or (lo is not None and v < lo) or (hi is not None and v > hi):
                ok = False
                break
        if ok:
            keep.append(col)
    return keep


def render_table():
    df, cols = state["df"], state["cols"]
    if df is None or not cols:
        table_html.value = ("" if df is None else
                            '<span style="font-size:12px;color:#a60">'
                            'no solution passes the filters</span>')
        return
    sel = state["selected"]
    head = "".join(
        f'<th style="width:{COL_W}px;text-align:right;padding:2px 6px;'
        f'border-bottom:1px solid #ccc;'
        f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">{html.escape(str(c))}</th>'
        for i, c in enumerate(cols))
    body = []
    for name in df.index:
        cells = "".join(
            f'<td style="width:{COL_W}px;text-align:right;padding:1px 6px;'
            f'font-variant-numeric:tabular-nums;'
            f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">'
            f'{_fmt(df.loc[name, c])}</td>'
            for i, c in enumerate(cols))
        body.append(
            f'<tr><td style="width:{ROW_LABEL_W}px;text-align:right;padding:1px 6px;'
            f'font-weight:600;">{html.escape(str(name))}</td>{cells}</tr>')
    table_html.value = ('<table style="border-collapse:collapse;font-size:12px;">'
                        f'<tr><td style="width:{ROW_LABEL_W}px"></td>{head}</tr>'
                        + "".join(body) + "</table>")


def select_solution(index):
    state["selected"] = index
    for b in solution_buttons.children:
        b.button_style = "info" if b._sel == index else ""
    render_table()
    push_view()


def rebuild_solution_buttons():
    buttons = []
    for i, col in enumerate(state["cols"]):
        b = widgets.Button(description=str(col), tooltip=f"show {col} in the 3D view",
                           layout=widgets.Layout(width=f"{COL_W}px", margin="0"))
        b.on_click(lambda _b, idx=i: select_solution(idx))
        b._sel = i
        buttons.append(b)
    # config sits in the row-label slot so the solution buttons stay aligned
    # with their table columns; zero goes after them
    config = widgets.Button(description="config",
                            tooltip="show the angle values entered on the left",
                            layout=widgets.Layout(width=f"{ROW_LABEL_W}px", margin="0"))
    config.on_click(lambda _b: select_solution(SEL_CONFIG))
    config._sel = SEL_CONFIG
    zero = widgets.Button(description="zero", tooltip="move all circles back to 0",
                          layout=widgets.Layout(width="60px", margin="0 0 0 8px"))
    zero.on_click(lambda _b: select_solution(SEL_ZERO))
    zero._sel = SEL_ZERO
    solution_buttons.children = [config] + buttons + [zero]


# --- the 3D view ------------------------------------------------------------

_html_path = REPO / "apps" / "diffractometer_3d.html"
_scene_html = _html_path.read_text()

view_frame = widgets.HTML(
    f'<iframe id="diff3d" srcdoc="{html.escape(_scene_html)}" '
    'style="width:100%; height:760px; border:1px solid #ddd;"></iframe>',
    layout=widgets.Layout(width="100%"),
)
payload_holder = widgets.HTML()
_push_count = {"n": 0}


def config_angles():
    """The six physical circles as entered on the left (view naming: gamma)."""
    return {("gamma" if n == "gam" else n): float(ctrl[n].value) for n in PHYSICAL_ANGLES}


def push_view():
    sel = state["selected"]
    angles = {n: 0.0 for n in ("mu", "eta", "chi", "phi", "delta", "gamma")}
    label = ""
    if sel == SEL_CONFIG:
        angles, label = config_angles(), "config"
    elif isinstance(sel, int) and state["df"] is not None and sel < len(state["cols"]):
        col = state["cols"][sel]
        angles = solution_angles(state["df"], col)
        label = str(col)
    msg = {"angles": angles, "kMag": k_magnitude(ctrl["energy"].value),
           "label": label, "n": _push_count["n"]}
    if state["ubcalc"] is not None:
        # the view derives the measured hkl itself, live, from UB + the angles
        msg["ub"] = reciprocal_basis(state["ubcalc"])
        # so the sample platelet shows the surface orientation actually entered
        msg["nPhi"] = surface_normal_phi(state["ubcalc"])
    vh = globals().get("view_height")      # exists once the shell is built
    if vh is not None:
        msg["frameHeight"] = int(vh.value)
    if state["display"] is not None:
        # sent once, so the view's own toggles are not reset on every push
        msg["display"] = state["display"]
        state["display"] = None
    _push_count["n"] += 1
    payload_holder.value = ('<span id="diff3d-payload" style="display:none">'
                            + html.escape(json.dumps(msg)) + "</span>")


# --- solve ------------------------------------------------------------------

def recompute(*_):
    global last_ubcalc, last_cons
    if _suspend["on"]:
        return
    ubcalc = None
    try:
        ubcalc = build_ubcalc(
            LAT_NAME,
            ctrl["uc_a"].value, ctrl["uc_b"].value, ctrl["uc_c"].value,
            ctrl["uc_alpha"].value, ctrl["uc_beta"].value, ctrl["uc_gamma"].value,
            [ln.strip() for ln in ctrl["orientations"].value.splitlines() if ln.strip()],
            ctrl["surface_normal"].value,
            frame=LAB_FRAME,
        )
        fixed = {n: ctrl[n].value for n in angle_names if ctrl[f"{n}_fixed"].value}
        cons = make_constraints(fixed)
        df = solve_hkl(ubcalc, cons, ctrl["h"].value, ctrl["k"].value,
                       ctrl["l"].value, ctrl["energy"].value)
    except Exception as exc:
        solve_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'
        state.update(df=None, cols=[])
        # a sample can be perfectly valid while this hkl/constraint set has no
        # solution; the view must still show the new cell, not the old one
        if ubcalc is not None:
            state["ubcalc"] = ubcalc
        if isinstance(state["selected"], int):
            state["selected"] = SEL_CONFIG
        rebuild_solution_buttons()
        select_solution(state["selected"])
        return

    # one standard row order, matching the input ordering
    ordered = [r for r in ROW_ORDER if r in df.index] + \
              [r for r in df.index if r not in ROW_ORDER]
    df = df.reindex(ordered)

    solve_msg.value = ""
    cols = passing_columns(df)
    state.update(df=df, cols=cols, ubcalc=ubcalc, cons=cons)
    last_ubcalc, last_cons = ubcalc, cons
    if isinstance(state["selected"], int) and state["selected"] >= len(cols):
        state["selected"] = SEL_CONFIG
    rebuild_solution_buttons()
    select_solution(state["selected"])


def _round_hkl(change):
    """Keep h, k, l to HKL_DECIMALS places.

    The correction is applied with solving suspended, so a rounded edit still
    costs exactly one solve rather than two.
    """
    rounded = round(float(change["new"]), HKL_DECIMALS)
    if rounded != change["new"]:
        _suspend["on"] = True
        try:
            change["owner"].value = rounded
        finally:
            _suspend["on"] = False


for _n in ("h", "k", "l"):
    ctrl[_n].observe(_round_hkl, names="value")

for _w in ctrl.values():
    _w.observe(recompute, names="value")
for _lo, _hi in filters.values():
    _lo.observe(recompute, names="value")
    _hi.observe(recompute, names="value")
recompute()


# --- tools (formerly separate cells) ----------------------------------------

from bernina_prep.diffraction import reflection_table

hkl_text = widgets.Textarea(
    value="1,0,0\n0,1,0\n0,0,1\n1,1,1",
    description="h,k,l list",
    layout=widgets.Layout(width="200px", height="120px"),
)
solution_idx = widgets.IntText(value=0, description="solution #", layout=widgets.Layout(width="190px"))
compute_btn = widgets.Button(description="Compute table", button_style="primary")
table_out = widgets.Output()


def parse_hkl_lines(text):
    reflections = []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        h, k, l = (float(x) for x in line.replace(";", ",").split(","))
        reflections.append((h, k, l))
    return reflections


def on_compute(_):
    table_out.clear_output()
    with table_out:
        if "last_ubcalc" not in globals():
            print("No valid sample/orientation yet.")
            return
        reflections = parse_hkl_lines(hkl_text.value)
        display(reflection_table(last_ubcalc, last_cons, reflections, ctrl["energy"].value, solution=solution_idx.value))


compute_btn.on_click(on_compute)
tool_table = widgets.VBox([widgets.HBox([hkl_text, solution_idx]), compute_btn, table_out])

from bernina_prep.diffraction import solve_hkl_positions, detector_hkl_grid, detector_hkl_summary

tdet_w = widgets.FloatText(value=500.0, description="tdet [mm]", layout=widgets.Layout(width="190px"))
det_w_w = widgets.FloatText(value=77.0, description="width [mm]", layout=widgets.Layout(width="190px"))
det_h_w = widgets.FloatText(value=77.0, description="height [mm]", layout=widgets.Layout(width="190px"))
grid_n_w = widgets.IntText(value=9, description="grid n", layout=widgets.Layout(width="190px"))
footprint_solution_idx = widgets.IntText(value=0, description="solution #", layout=widgets.Layout(width="190px"))
footprint_btn = widgets.Button(description="Compute Q range", button_style="primary")
plot_btn = widgets.Button(description="Show 3D plot (slower)", button_style="")
footprint_out = widgets.Output()
plot_out = widgets.Output()

_last_footprint = {}


def _current_position():
    h, k, l = ctrl["h"].value, ctrl["k"].value, ctrl["l"].value
    energy = ctrl["energy"].value
    result = solve_hkl_positions(last_ubcalc, last_cons, h, k, l, energy)
    idx = footprint_solution_idx.value if footprint_solution_idx.value < len(result) else len(result) - 1
    pos, _ = result[idx]
    return pos, energy


def on_footprint(_):
    footprint_out.clear_output()
    with footprint_out:
        if "last_ubcalc" not in globals():
            print("No valid sample/orientation yet.")
            return
        pos, energy = _current_position()
        H, K, L = detector_hkl_grid(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        _last_footprint.update(pos=pos, energy=energy, H=H, K=K, L=L)
        summary = detector_hkl_summary(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        print(f"Detector center at delta={pos.delta:.3f}, gamma={pos.nu:.3f}, tdet={tdet_w.value:.1f} mm")
        display(summary.to_frame("value"))


def on_plot(_):
    plot_out.clear_output()
    with plot_out:
        if not _last_footprint:
            print("Compute the Q range first.")
            return
        import matplotlib.pyplot as plt

        H, K, L = _last_footprint["H"], _last_footprint["K"], _last_footprint["L"]
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(projection="3d")
        ax.plot_surface(H, K, L, alpha=0.6)
        ax.set_xlabel("h")
        ax.set_ylabel("k")
        ax.set_zlabel("l")
        n = H.shape[0] // 2
        cx, cy, cz = H[n, n], K[n, n], L[n, n]
        arrowlen = 0.3 * max(H.max() - H.min(), K.max() - K.min(), L.max() - L.min(), 1e-6)
        ax.quiver(cx, cy, cz, arrowlen, 0, 0, color="r")
        ax.quiver(cx, cy, cz, 0, arrowlen, 0, color="g")
        ax.quiver(cx, cy, cz, 0, 0, arrowlen, color="b")
        plt.show()


footprint_btn.on_click(on_footprint)
plot_btn.on_click(on_plot)
tool_footprint = widgets.VBox([
    widgets.HBox([tdet_w, det_w_w, det_h_w], layout=widgets.Layout(flex_flow="row wrap")),
    widgets.HBox([grid_n_w, footprint_solution_idx], layout=widgets.Layout(flex_flow="row wrap")),
    widgets.HBox([footprint_btn, plot_btn]),
    footprint_out,
    plot_out,
])

# --- app shell: panels the user can arrange -----------------------------------
# Everything above builds the widgets and the solving logic; this part only
# places them. Each panel is a card with a collapse button; the layout switch
# re-parents the same panel widgets into a different arrangement, so nothing
# is rebuilt and no state is lost. Sizes stay in px (see CLAUDE.md: percentage
# sizes can collapse a column under voila).

INPUT_W = "404px"


class Panel:
    """A titled card whose body can be collapsed."""

    def __init__(self, key, title, body, is_open=True):
        self.key = key
        self.body = body
        self.toggle = widgets.Button(description="▾", tooltip="collapse / expand",
                                     layout=widgets.Layout(width="30px", height="26px",
                                                           padding="0", margin="0 6px 0 0"))
        self.toggle.on_click(lambda _b: self.set_open(not self.is_open))
        head = widgets.HBox([self.toggle, widgets.HTML(f"<b>{title}</b>")],
                            layout=widgets.Layout(align_items="center"))
        self.box = widgets.VBox([head, body], layout=widgets.Layout(
            border="1px solid #d0d7de", padding="6px 8px", margin="0 8px 8px 0"))
        self.is_open = True
        self.set_open(is_open)

    def set_open(self, value):
        self.is_open = bool(value)
        self.body.layout.display = "flex" if self.is_open else "none"
        self.toggle.description = "▾" if self.is_open else "▸"


# inputs, as tabs; the six physical circles stay in view, the virtual
# constraints fold away
physical_rows = angle_rows[:len(PHYSICAL_ANGLES)]
virtual_rows = angle_rows[len(PHYSICAL_ANGLES):]
more_constraints = widgets.Accordion(children=[widgets.VBox(virtual_rows)],
                                     titles=("more constraints (virtual angles)",),
                                     selected_index=None)
input_tabs = widgets.Tab(children=[
    widgets.VBox([ctr_uc, ctr_or]),
    widgets.VBox([ctr_hkl, energy_row]),
    widgets.VBox([filter_header] + physical_rows + [more_constraints]),
    widgets.VBox([preset_dd, preset_msg, widgets.HBox([store_btn, upload]),
                  store_panel, config_msg]),
], titles=("sample", "reflection", "angles", "config"), selected_index=1,
    layout=widgets.Layout(width=INPUT_W))

tool_tabs = widgets.Tab(children=[tool_table, tool_footprint],
                        titles=("table of reflections", "detector footprint"))

panels = {
    "inputs": Panel("inputs", "inputs", input_tabs),
    # short titles: a collapsed card is only as wide as its header
    "solutions": Panel("solutions", "solutions", widgets.VBox([
        widgets.HTML('<span style="font-size:11px;color:#666">click a column to show '
                     'it in the 3D view</span>'),
        solve_msg, solution_buttons, table_html])),
    "view": Panel("view", "3D view", widgets.VBox([view_frame, payload_holder])),
    "tools": Panel("tools", "tools", tool_tabs, is_open=False),
}

LAYOUTS = {"side by side": "side", "large 3D": "large", "stacked": "stacked"}
layout_switch = widgets.ToggleButtons(options=list(LAYOUTS), value="large 3D",
                                      tooltips=["inputs | solutions | 3D in one row (wraps when narrow)",
                                                "3D view large, inputs and solutions beside it",
                                                "inputs and solutions on top, 3D full width below"],
                                      style={"button_width": "96px"})
view_height = widgets.BoundedIntText(value=760, min=300, max=1600, step=20,
                                     description="3D height",
                                     layout=widgets.Layout(width="150px"),
                                     style={"description_width": "62px"})
help_btn = widgets.ToggleButton(value=False, description="help", icon="question",
                                layout=widgets.Layout(width="80px"))
help_box = widgets.HTML(HELP_HTML, layout=widgets.Layout(
    display="none", border="1px solid #d0d7de", padding="4px 12px", margin="0 0 8px 0"))
help_btn.observe(lambda c: setattr(help_box.layout, "display", "flex" if c["new"] else "none"),
                 names="value")

header = widgets.HBox(
    [widgets.HTML('<span style="font-size:15px;font-weight:600">Diffraction reflections '
                  '<span style="font-weight:400;color:#666">&middot; Bernina, you geometry</span></span>',
                  layout=widgets.Layout(margin="0 16px 0 0")),
     layout_switch, view_height, help_btn],
    layout=widgets.Layout(align_items="center", flex_flow="row wrap", margin="0 0 8px 0"))

# a column container: it stretches each arrangement to the full width, so the
# rows inside can wrap and the 3D panel can grow
body = widgets.VBox(layout=widgets.Layout(width="100%"))


def arrange(change=None):
    mode = LAYOUTS[layout_switch.value]
    p = {k: v.box for k, v in panels.items()}
    p["view"].layout.flex = "1 1 640px" if mode == "large" else "1 1 520px"
    p["view"].layout.min_width = "0"
    for k in ("inputs", "solutions", "tools"):
        p[k].layout.flex = "0 0 auto"
    if mode == "side":
        row = widgets.HBox([p["inputs"], p["solutions"], p["view"]],
                           layout=widgets.Layout(flex_flow="row wrap", align_items="flex-start"))
        body.children = [widgets.VBox([row, p["tools"]])]
    elif mode == "large":
        left = widgets.VBox([p["inputs"], p["solutions"], p["tools"]],
                            layout=widgets.Layout(flex="0 0 auto"))
        body.children = [widgets.HBox([left, p["view"]],
                                      layout=widgets.Layout(flex_flow="row wrap",
                                                            align_items="flex-start"))]
    else:
        top = widgets.HBox([p["inputs"], p["solutions"], p["tools"]],
                           layout=widgets.Layout(flex_flow="row wrap", align_items="flex-start"))
        body.children = [widgets.VBox([top, p["view"]])]


def _view_controls_for_layout(change):
    # wide view: put the 3D page's own controls beside the plot, otherwise
    # under it. Sent once per switch, so the page's own toggle still works.
    state["display"] = {"controlsBelow": LAYOUTS[change["new"]] != "large"}
    push_view()


layout_switch.observe(arrange, names="value")
layout_switch.observe(_view_controls_for_layout, names="value")
view_height.observe(lambda _c: push_view(), names="value")
arrange()
# the view starts in the default arrangement's control placement
state["display"] = {"controlsBelow": LAYOUTS[layout_switch.value] != "large"}


def layout_state():
    return {"arrangement": layout_switch.value,
            "open": {k: v.is_open for k, v in panels.items()},
            "input_tab": input_tabs.selected_index,
            "view_height": view_height.value}


def apply_layout(cfg):
    if cfg.get("arrangement") in LAYOUTS:
        layout_switch.value = cfg["arrangement"]
    for k, is_open in (cfg.get("open") or {}).items():
        if k in panels:
            panels[k].set_open(is_open)
    if isinstance(cfg.get("input_tab"), int):
        input_tabs.selected_index = cfg["input_tab"]
    if isinstance(cfg.get("view_height"), int):
        view_height.value = cfg["view_height"]


push_view()
display(widgets.VBox([header, help_box, body], layout=widgets.Layout(width="100%")))
